In [1]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
import itertools as itools
import time
import numpy as np
from collections import defaultdict
from scipy.optimize import linprog
import chaospy as cp
from typing import List, Callable, Union
from numpy.polynomial.legendre import leggauss
import pandas as pd
from pyomo.environ import *

In [2]:
def mpformulate_theta_bounds(flex_sol, num_theta: int, theta_bounds: list, num_design: int = 0, design_bounds: list = None, psi_idx: int = 0, theta_m: int = 0):
    A0, b0, F0 = np.empty((len(flex_sol), num_theta)), np.empty(
        (len(flex_sol), 1)), np.empty((len(flex_sol), num_design))
    num_cr = len(flex_sol.critical_regions)
    for i, region in enumerate(flex_sol.critical_regions):
        A0[i] = region.A[psi_idx, :num_theta]
        b0[i] = -region.b[psi_idx]
        F0[i] = -region.A[psi_idx, num_theta:num_theta+num_design]
    # print(f'num_cr:{num_cr}')
    # print(f'num_theta:{num_theta}')
    # print(f'num_design:{num_design}')
    # print(f"A0: {A0}")
    # print(f"b0: {b0}")
    # print(f"F0: {F0}")

    c = np.hstack([np.array([-1, 1]).reshape(1, -1),
                  np.zeros((1, 2 * (num_theta - 1 - theta_m)))]).reshape(-1, 1)
    # print(f'c:{c}')
    # print(f'c.shape: {c.shape}')

    row1_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (A0[:, [i]], np.zeros((num_cr, 1)))])
    row2_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (np.zeros((num_cr, 1)), A0[:, [i]])])
    bound_row = np.hstack([np.array([-1, 1]).reshape(1, -1),
                          np.zeros((1, 2 * (num_theta - 1 - theta_m)))])
    A = np.vstack([row1_block, row2_block, bound_row, -
                  np.eye(2*(num_theta-theta_m)), np.eye(2*(num_theta-theta_m))])
    # print(f'A: {A}')
    # print(f'A.shape: {A.shape}')

    x_lb = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][0]] * 2])
    x_ub = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][1]] * 2])
    b = np.vstack([b0, b0, np.zeros((1, 1)), -
                  x_lb.reshape(-1, 1), x_ub.reshape(-1, 1)])
    # print(f'b: {b}')
    # print(f'b.shape: {b.shape}')

    if F0.size == 0 and theta_m == 0:
        # print('here')
        return A, b, c, np.array([]), np.array([]), np.array([]), np.array([])

    F = np.vstack([F0, F0, np.zeros((1, num_design)), np.zeros(
        (4*(num_theta-theta_m), num_design))]) if num_design > 0 else np.vstack([F0, F0])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    if theta_m > 0:
        F_lltheta = np.hstack([A0[:, [i]] for i in range(theta_m)])
        # print(f'F_lltheta: {F_lltheta}')
        # print(f'F_lltheta.shape: {F_lltheta.shape}')
        F = np.hstack([np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))]), F]
                      ) if F.size > 0 else np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')

    H = np.zeros((2*(num_theta-theta_m), theta_m+num_design))
    # print(f'H:{H}')
    # print(f'H.shape: {H.shape}')

    A_t = np.vstack([-np.eye(theta_m+num_design), np.eye(theta_m+num_design)])
    # print(f'A_t:{A_t}')
    # print(f'A_t.shape: {A_t.shape}')

    theta_lb = np.array([-theta_bounds[i][0] for i in range(theta_m)] + ([-j[0] for j in design_bounds] if isinstance(design_bounds, list)
                                                                         else [])).reshape(-1, 1)
    theta_ub = np.array([theta_bounds[i][1] for i in range(theta_m)] + ([j[1] for j in design_bounds] if isinstance(design_bounds, list)
                                                                        else [])).reshape(-1, 1)

    b_t = np.vstack([theta_lb, theta_ub])
    # print(f'b_t:{b_t}')
    # print(f'b_t.shape: {b_t.shape}')

    return A, b, c, H, A_t, b_t, F

In [3]:
def get_theta_bounds(flex_sol, numt, tbounds, numd: int = 0, dbounds: list = None):
    theta_bound_dict = defaultdict(dict)
    prob_dict = defaultdict(dict)
    
    if isinstance(tbounds, dict):
        tbounds = [tbounds[k] for k in tbounds]
    # print(f'tbounds: {tbounds}')
    
    if isinstance(dbounds, dict):
        dbounds = [dbounds[k] for k in dbounds]
    # print(f'dbounds: {dbounds}')
    
    for i in range(numt):
        A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(
            flex_sol=flex_sol, num_theta=numt, num_design=numd, theta_bounds=tbounds, design_bounds=dbounds, theta_m=i)
        # print(f'A.shape:{A.shape}')
        # print(f'b.shape: {b.shape}')
        # print(f'F.shape: {F.shape}')
        if F.size != 0:
            prob = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)
            prob.process_constraints()
            solution = solve_mpqp(
                problem=prob, algorithm=mpqp_algorithm.combinatorial_parallel)
            prob_dict[f't{i}'] = prob
            theta_bound_dict[f't{i}'] = solution
        else:
            linsol = linprog(c=c, A_ub=A, b_ub=b)
            prob_dict[f't{i}'] = linsol
            theta_bound_dict[f't{i}'] = [linsol.x[1], linsol.x[0]]
            # if linsol.success:
            # print("Optimal value:", linsol.fun)
            # print("Optimal x:", linsol.x)
        print(f'Finished solving for theta{i+1}')
    probs = [p for key, p in prob_dict.items()]
    sols = [sol for key, sol in theta_bound_dict.items()]

    return probs, sols

In [4]:
def gauss_legendre_between_bounds(expr_coeffs: np.ndarray, n_gl: int, max_idx: int = 0, min_idx: int = 1):
    """
    Generate n Gauss–Legendre quadrature points and weights between min and max bounds
    defined by two linear expressions.

    Parameters:
        expr_coeffs (np.ndarray): 2xD array. Row 0 = max point co`efficients, Row 1 = min.
        n (int): Number of quadrature points.

    Returns:
        points (np.ndarray): (n, D) array of quadrature points.
        weights (np.ndarray): (n,) array of weights.
    """
    if expr_coeffs.shape[0] != 2:
        raise ValueError("expr_coeffs must have two rows")

    max_coeffs = expr_coeffs[max_idx]
    min_coeffs = expr_coeffs[min_idx]

    # Get Gauss–Legendre points and weights on [-1, 1]
    nodes, weights = leggauss(n_gl)
    weights = weights.reshape(-1, 1)

    # Affine transformation to domain [min_coeffs, max_coeffs]
    points = 0.5 * (np.outer((nodes + 1), max_coeffs) +
                    np.outer((1 - nodes), min_coeffs))

    # Adjust weights to match new domain
    weights = 0.5 * weights@(max_coeffs - min_coeffs).reshape(1, -1)

    return points, weights


def get_quadrature_points(solution, nq: int, t_vector: np.ndarray):
    # Augment t_vector once
    t_vector_aug = np.append(t_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        qpoints, qweights = np.polynomial.legendre.leggauss(nq)
        min, max = solution[0], solution[1]
        qps_mapped = 0.5*(max*(1+qpoints) + min*(1-qpoints))
        qws_mapped = 0.5*(max-min)*qweights
        # print(max, min, qps_mapped, qws_mapped)
        return max, min, qps_mapped, qws_mapped

    for region in solution.critical_regions:
        if region.is_inside(t_vector.reshape(-1, 1)):
            coeffs = np.concatenate([region.A, region.b], axis=1)[:2, :]
            qpoints, qweights = gauss_legendre_between_bounds(
                expr_coeffs=coeffs, n_gl=nq)
            return coeffs[0] @ t_vector_aug, coeffs[1] @ t_vector_aug, qpoints @ t_vector_aug, qweights @ t_vector_aug

    # print(f't_vector: {t_vector}')
    # print(f'solution:{solution}')
    raise ValueError("No region found that contains the given t_vector.")


def calculate_stocflexibility(
    sols,
    nq: Union[int, list],
    joint_func,
    d_vector: np.ndarray = None,
    verbose: bool = True,
):
    # Validate nq if it's a list
    if isinstance(nq, list):
        if len(nq) != len(sols):
            raise ValueError(
                "If nq is a list, it must have the same length as sols")

    start_time = time.perf_counter()

    def recurse(level: int, theta_prev: list, weight_prev: float) -> float:
        if level == len(sols):
            return weight_prev * joint_func(theta_prev)

        nql = nq[level] if isinstance(nq, list) else nq

        t_vector = (
            np.block([np.array(theta_prev), d_vector])
            if isinstance(d_vector, np.ndarray)
            else np.array(theta_prev)
        )
        
        # print(f"t_vector: {t_vector}")
        
        _, _, t_points, t_weights = get_quadrature_points(
            solution=sols[level],
            nq=nql,
            t_vector=t_vector.reshape(-1,1),
        )

        t_points = t_points.flatten()
        t_weights = t_weights.flatten()

        return sum(
            recurse(level + 1, theta_prev + [v], weight_prev * w)
            for v, w in zip(t_points, t_weights)
        )

    stflex = recurse(level=0, theta_prev=[], weight_prev=1.0)

    end_time = time.perf_counter()

    if verbose:
        print(f"Gaussian Legendre Stochastic Flexibility Elapsed time: {end_time - start_time:.4f} s"
              )

    return stflex

In [5]:
def theta_interval_at_point(solution, theta_vector: np.ndarray, max_idx: int = 0, min_idx: int = 1) -> tuple:
    """Given the parametric solution for theta_k and the current 'state' vector (theta_prev + d),
        return the scalar lower and upper bound [t_min, t_max] for this theta_k.

    Args:
        solution (_type_): _description_
        t_vector (np.ndarray): _description_
        max_idx (int, optional): _description_. Defaults to 0.
        min_idx (int, optional): _description_. Defaults to 1.

    Returns:
        tuple: _description_
    """

    theta_vector_aug = np.append(theta_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        theta_min = solution[0]
        theta_max = solution[1]
        return float(theta_min), float(theta_max)

    for region in solution.critical_regions:
        if region.is_inside(theta_vector.reshape(-1, 1)):
            coefficients = np.concatenate([region.A, region.b], axis=1)[:2, :]
            max_coefficients = coefficients[max_idx]
            min_coefficients = coefficients[min_idx]
            theta_max = (max_coefficients @ theta_vector_aug).item()
            theta_min = (min_coefficients @ theta_vector_aug).item()
            return theta_min, theta_max

    raise ValueError(
        "The provided theta_vector is not inside any critical region of the solution.")


def map_u_to_theta_and_jacobian(solutions: List, u: np.ndarray, d_vector: np.ndarray = None) -> tuple:
    """Given the parametric solutions for theta_k and the current 'state' vector (theta_prev + d),
        return the theta_k vector and the Jacobian matrix dtheta/du.
    Args:
        solutions (List): List of parametric solutions for each theta_k.
        u (np.ndarray): 1D arraay of canonical coordinates 
        d_vector (np.ndarray): Current disturbance vector.
    """
    theta_values = []
    jacobian = 1.0

    for k, sol in enumerate(solutions):
        if isinstance(d_vector, np.ndarray):
            theta_vector = np.block([np.array(theta_values), d_vector])
        else:
            theta_vector = np.array(theta_values, dtype=float)

        theta_min, theta_max = theta_interval_at_point(sol, theta_vector)
        length = theta_max - theta_min
        
        theta_k = 0.5 * length * u[k] + 0.5 * (theta_max + theta_min)
        theta_values.append(theta_k)

        jacobian *= 0.5 * length

    return np.array(theta_values, dtype=float), jacobian


def calculate_stocflexibility_smolyak(solutions: List, level: int, joint_func: Callable[[List[float]], float], d_vector: np.ndarray = None, rule: str = "gaussian") -> float:
    """Compute stochastic flexibility using a Smolyak sparse grid in canonical  u-space

    Args:
        solutions (List): list of solutions for each theta dimension (same structure as in calculate_stocflexibility)
        level (int): Smolyak level (1,2,3,...) controls accuracy & number of points
        joint_func (Callable[[List[float]], float]): callable f(theta_list) -> scalar
        d_vector (np.ndarray, optional):design vector (np.ndarray). Defaults to None.
        rule (str, optional): 1D quadrature rule passed to chaospy (e.g. "gaussian"). Defaults to "gaussian".

    Returns:
        float: _description_
    """

    n_theta = len(solutions)

    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])

    nodes_u, weights_expectation = cp.quadrature.sparse_grid(
        order=level, dist=dist, rule=rule)

    weights_u = weights_expectation * (2.0 ** n_theta)

    nodes_u = nodes_u.T

    start = time.time()
    stochastic_flexibility = 0.0

    for i in range(nodes_u.shape[0]):
        u_vector = nodes_u[i, :]
        theta_vector, jacobian = map_u_to_theta_and_jacobian(
            solutions, u_vector, d_vector)
        func_value = joint_func(theta_vector)
 
        stochastic_flexibility += func_value * jacobian * weights_u[i]

    end = time.time()
    print(
        f"Smolyak stochastic flexibility computed in {end - start:.4f} seconds.")
    return stochastic_flexibility

In [6]:
# m = ConcreteModel()
# 
# # y = {'0':1, '1':1, '2':1, '3':1}
# # y = {'0':0.5, '1':0.5, '2':0.5, '3':0.5}
# y = {'0':0, '1':0, '2':0, '3':0}
# 
# suppliers = ['A', 'B', 'C']
# markets = ['D', 'E', 'F']
# sites = ['0', '1', '2', '3']
# 
# supply_theta = {'A': 45000, 'B': 100000, 'C': 75000}
# 
# conv = 1/2.4 # basis is feed
# 
# demand = {'D': 0, 'E': 60000, 'F': 25000} # demand of product
# 
# no_trips = 365 # number of trips made by each truck per year, basically one round trip per day per truck
# 
# site_cap = {'0': 50000, '1': 50000, '2': 50000, '3': 75000} # capacity of each site, basis is feed
# 
# flex_goal = 0.17 # goal for cfri
# 
# availability_factor = 0.959260199
# 
# m.suppliers = Set(initialize=suppliers)
# m.markets = Set(initialize=markets)
# m.sites = Set(initialize=sites)
# 
# # control variables
# m.product = Var(m.sites, domain = NonNegativeReals, doc = 'amount of product made at each site')
# m.feed = Var(m.sites, domain = NonNegativeReals, doc = 'amount of feed used at each site')
# m.feed_ship = Var(m.suppliers, m.sites, domain = NonNegativeReals, doc = 'amount of feed shipped from supplier to site')
# m.product_ship = Var(m.sites, m.markets, domain = NonNegativeReals, doc = 'amount of product shipped from site to market')
# 
# # design variables
# m.trans_cap = Var(m.sites, domain = NonNegativeReals, doc = 'transport capacity around each site')
# m.capex = Var(domain = NonNegativeReals)
# 
# # CFRI
# # m.cfri = Var(domain = NonNegativeReals)
# # 
# # constraints
# def balance_feed(model, site):
#     return model.feed[site] == sum(model.feed_ship[supplier, site] for supplier in model.suppliers)
# m.balance_feed_con = Constraint(m.sites, rule=balance_feed)
# 
# def conversion(model, site):
#     return model.product[site] == model.feed[site]*conv
# m.conversion_con = Constraint(m.sites, rule=conversion)
# 
# def balance_product(model, site):
#     return model.product[site] == sum(model.product_ship[site, market] for market in model.markets)
# m.balance_product_con = Constraint(m.sites, rule=balance_product)
# 
# def transport_cost(model):
#     return model.capex == sum(model.trans_cap[site] for site in model.sites)
# m.transport_cost_con = Constraint(rule=transport_cost)
# 
# def limit_feed(model, supplier):
#     return sum(model.feed_ship[supplier, site] for site in model.sites) - supply_theta[supplier] <= 0
# m.limit_feed_con = Constraint(m.suppliers, rule=limit_feed)
# 
# def limit_production(model, site):
#     return model.feed[site] - site_cap[site]*availability_factor <=0
# m.limit_production_con = Constraint(m.sites, rule=limit_production)
# 
# def limit_demand(model, market):
#     return sum(model.product_ship[site, market] for site in model.sites) >= demand[market]
# m.limit_demand_con = Constraint(m.markets, rule=limit_demand)
# 
# def limit_transport(model, site):
#     return sum(model.feed_ship[supplier, site] for supplier in model.suppliers) + sum(model.product_ship[site, market] for market in markets) - model.trans_cap[site] * no_trips * y[site] <= 0
# m.limit_transport_con = Constraint(m.sites, rule=limit_transport)
# 
# # def cfri_rule(model):
# #     d0_ratio = model.trans_cap[sites[0]]/576
# #     d1_ratio = model.trans_cap[sites[1]]/576
# #     d2_ratio = model.trans_cap[sites[2]]/576
# #     d3_ratio = model.trans_cap[sites[3]]/576
# # 
# #     return coefs[0] + coefs[1]*d0_ratio + coefs[2]*d1_ratio + coefs[3]*d2_ratio + coefs[4]*d3_ratio + coefs[5]*d0_ratio**2 + coefs[6]*d0_ratio*d1_ratio + coefs[7]*d0_ratio*d2_ratio + coefs[8]*d0_ratio*d3_ratio + coefs[9]*d1_ratio**2 + coefs[10]*d1_ratio*d2_ratio + coefs[11]*d1_ratio*d3_ratio + coefs[12]*d2_ratio**2 + coefs[13]*d2_ratio*d3_ratio + coefs[14]*d3_ratio**2 + intercept == model.cfri 
# # m.cfri_con = Constraint(rule=cfri_rule)
# 
# # def flex_rule(model):
# #     return model.cfri >= flex_goal
# # m.flex_con = Constraint(rule=flex_rule)
# 
# # objective
# m.obj = Objective(expr = (m.capex), sense=minimize)
# 
# # solve
# results = SolverFactory('gurobi', solver_io = 'python').solve(m, tee = True)

In [7]:
suppliers = ['A', 'B', 'C']
markets = ['D', 'E', 'F']
sites = ['0', '1', '2', '3']

supply_theta_nominal = {'A': 45000, 'B': 100000, 'C': 75000}
supply_theta_stddev = {'A': np.sqrt(7500), 'B': np.sqrt(16667), 'C': np.sqrt(12500)}

conv = 1/2.4 # basis is feed

demand = {'D': 0, 'E': 60000, 'F': 25000} # demand of product

no_trips = 365 # number of trips made by each truck per year, basically one round trip per day per truck

site_cap = {'0': 50000, '1': 50000, '2': 50000, '3': 75000} # capacity of each site, basis is feed

flex_goal = 0.17 # goal for cfri

availability_factor = 0.959260199

In [8]:
t_bounds = {'A':(supply_theta_nominal['A'] - 4*supply_theta_stddev['A'], supply_theta_nominal['A'] + 4*supply_theta_stddev['A']),
            'B':(supply_theta_nominal['B'] - 4*supply_theta_stddev['B'], supply_theta_nominal['B'] + 4*supply_theta_stddev['B']),
            'C':(supply_theta_nominal['C'] - 4*supply_theta_stddev['C'], supply_theta_nominal['C'] + 4*supply_theta_stddev['C'])}

d_bounds = {'0':(0, 1000), '1':(0, 1000), '2':(0, 1000), '3':(0, 1000)}
nt = len(t_bounds)
nd = len(d_bounds)

In [60]:
y = {'0':1, '1':1, '2':1, '3':1}
# y = {'0':0.5, '1':0.5, '2':0.5, '3':0.5}
# y = {'0':0, '1':0, '2':0, '3':0}

In [61]:
m = MPModeler()

In [62]:
u = m.add_var(name='u')
product = {si: m.add_var(name=f'product[{si}]') for si in sites}
feed = {si: m.add_var(f'feed[{si}]') for si in sites}
feed_ship = {(su, si): m.add_var(name=f'feed_ship[{su},{si}]') for su, si in itools.product(suppliers, sites)}
product_ship = {(si, ma): m.add_var(name=f'product_ship[{si},{m}]') for si, ma in itools.product(sites, markets)}
capex = m.add_var(name='capex')

supply_theta = {su: m.add_param(name=f'supply_theta[{su}]') for su in suppliers}

trans_cap = {si: m.add_param(name=f'trans_cap[{si}]') for si in sites}

In [63]:
# def balance_feed(model, site):
#     return model.feed[site] == sum(model.feed_ship[supplier, site] for supplier in model.suppliers)
# m.balance_feed_con = Constraint(m.sites, rule=balance_feed)

m.add_constrs(sum(feed_ship[su, si] for su in suppliers) == feed[si] for si in sites)

In [64]:
# def transport_cost(model):
#     return model.capex == sum(model.trans_cap[site] for site in model.sites)
# m.transport_cost_con = Constraint(rule=transport_cost)

m.add_constr(capex == sum(trans_cap[si] for si in sites))

In [65]:
# def conversion(model, site):
#     return model.product[site] == model.feed[site]*conv
# m.conversion_con = Constraint(m.sites, rule=conversion)

m.add_constrs(product[si] == feed[si]*conv for si in sites)

In [66]:
# def balance_product(model, site):
#     return model.product[site] == sum(model.product_ship[site, market] for market in model.markets)
# m.balance_product_con = Constraint(m.sites, rule=balance_product)

m.add_constrs(sum(product_ship[si, ma] for ma in markets) == product[si] for si in sites)

In [67]:
# def limit_feed(model, supplier):
#     return sum(model.feed_ship[supplier, site] for site in model.sites) - supply_theta[supplier] <= 0
# m.limit_feed_con = Constraint(m.suppliers, rule=limit_feed)

m.add_constrs(sum(feed_ship[su, si] for si in sites) <= supply_theta[su] + u for su in suppliers)

In [68]:
# def limit_production(model, site):
#     return model.feed[site] - site_cap[site] <=0
# m.limit_production_con = Constraint(m.sites, rule=limit_production)

m.add_constrs(feed[si] <= site_cap[si] * availability_factor + u for si in sites)

In [69]:
# def limit_demand(model, market):
#     return sum(model.product_ship[site, market] for site in model.sites) >= demand[market]
# m.limit_demand_con = Constraint(m.markets, rule=limit_demand)

m.add_constrs(demand[ma] <= sum(product_ship[si, ma] for si in sites) + u for ma in markets)

In [70]:
# def limit_transport(model, site):
#     return sum(model.feed_ship[supplier, site] for supplier in model.suppliers) + sum(model.product_ship[site, market] for market in markets) - model.trans_cap[site] * no_trips <= 0
# m.limit_transport_con = Constraint(m.sites, rule=limit_transport)

m.add_constrs(feed[si] + product[si] <= trans_cap[si]*no_trips*y[si] + u for si in sites)

In [71]:
# theta bounds
m.add_constrs(supply_theta[su] >= t_bounds[su][0] for su in suppliers)
m.add_constrs(supply_theta[su] <= t_bounds[su][1] for su in suppliers)

In [72]:
# design bounds
m.add_constrs(trans_cap[si] >= d_bounds[si][0] for si in sites)
m.add_constrs(trans_cap[si] <= d_bounds[si][1] for si in sites)

In [73]:
m.set_objective(u)

In [74]:
prob = m.formulate_problem()
prob.process_constraints()

In [75]:
# np.hstack((prob.A_t, prob.b_t))

In [76]:
solution_flexibility = solve_mpqp(problem=prob, algorithm=mpqp_algorithm.geometric_parallel)

Using a found active set [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22]
Spawned threads across 24
 Number of Facets to look at this time 14
 Number of Facets to look at this time 56
 Number of Facets to look at this time 84
 Number of Facets to look at this time 56
 Number of Facets to look at this time 14


In [77]:
len(solution_flexibility.critical_regions)

16

In [78]:
start_time = time.time()
prob_list, sol_list = get_theta_bounds(flex_sol=solution_flexibility, numt=nt, numd=nd, tbounds=t_bounds, dbounds=d_bounds)
end_time = time.time()
print(f'Elapsed time for solving mp problems: {end_time-start_time}')

Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 13
Time to run all tasks in parallel 0.015244245529174805
Time to process all depth outputs 0.0
Time at depth test 2, 0.015244245529174805
Number of active sets to be considered is 36
Time to run all tasks in parallel 0.04122495651245117
Time to process all depth outputs 0.0
Time at depth test 3, 0.05646920204162598
Number of active sets to be considered is 116
Time to run all tasks in parallel 0.06378746032714844
Time to process all depth outputs 0.0
Time at depth test 4, 0.12025666236877441
Number of active sets to be considered is 220
Time to run all tasks in parallel 0.09377050399780273
Time to process all depth outputs 0.0
Time at depth test 5, 0.21402716636657715
Number of active sets to be considered is 314
Time to run all tasks in parallel 0.1326885223388672
Time to process all depth outputs 0.0
Time at depth test 6, 0.34671568870544434
Number of active sets to be considered is 312
Tim

In [79]:
len(sol_list[0])

16

In [80]:
len(sol_list[1])

4

In [81]:
len(sol_list[2])

1

In [82]:
def joint_pdf(theta: list):
    return ((1/np.sqrt(2*np.pi)) * (1/supply_theta_stddev['A']) * np.exp(-0.5 * ((theta[0] - supply_theta_nominal['A'])/supply_theta_stddev['A'])**2) *
            (1/np.sqrt(2*np.pi)) * (1/supply_theta_stddev['B']) * np.exp(-0.5 * ((theta[1] - supply_theta_nominal['B'])/supply_theta_stddev['B'])**2) *
            (1/np.sqrt(2*np.pi)) * (1/supply_theta_stddev['C']) * np.exp(-0.5 * ((theta[2] - supply_theta_nominal['C'])/supply_theta_stddev['C'])**2))

In [83]:
# d_vector = np.array([194, 132, 175, 291])
d_vector = np.array([388, 405, 402, 419])
# d_vector = np.array([460, 537, 481, 469])
nq = 12

sf_idx_gaussian = calculate_stocflexibility(sols=sol_list, nq=nq, joint_func=joint_pdf, d_vector=d_vector)
sf_idx_smolyak = calculate_stocflexibility_smolyak(solutions=sol_list, level=20, joint_func=joint_pdf, d_vector=d_vector)

print(f'Stochastic Flexibility Index Gaussian Legendre quadrature: {sf_idx_gaussian:.4}')
print(f'Stochastic Flexibility Index Smolyak quadrature: {sf_idx_smolyak:.4}')

Gaussian Legendre Stochastic Flexibility Elapsed time: 0.0497 s
Smolyak stochastic flexibility computed in 6.0522 seconds.
Stochastic Flexibility Index Gaussian Legendre quadrature: 0.9998
Stochastic Flexibility Index Smolyak quadrature: 0.9998


In [84]:
# cr_idx = 1

In [85]:
# np.hstack((sol_list[cr_idx].critical_regions[cr_idx].E, sol_list[cr_idx].critical_regions[cr_idx].f))